# Pooling Layers

Pooling layers are layers that popularly used in deep learning models for computer vision task. The task that these layers perform is called **under sampling** or **down sampling** or **sub sampling**. The concept of down sampling was introduced by Yann LeCunn and other researchers when they created the first CNN model called **LeNet**. Down sampling was one of the principles introduced in LeNet paper.

Before continuing ahead I recommend reading about it in detail over here [LeNet Paper Discussion](https://github.com/Vaibhavsareen1/computer-vision/blob/main/notebooks/lenet_1.ipynb). It will help you understand the role pooling layers play.

In CNN models built for computer vision tasks the pooling layers are found in the feature extractor part of the models. The feature extractor models generally contain repeating blocks of **convolution layers** followed by **non linear activation functions** followed by **pooling layers**.

Pooling layers are also convolution functions but unlike **Convolution** neural network layers they do not have any weights and biases associated with them. These layers are there to reduce the size of the incoming features at the same time retaining the most critical information from it's receptive field. This decreases complexity and reduces the chance of overfitting.

There are main two types of pooling layers used<br>
1. **Max Pooling Layer**: Takes the maximum value present in it's receptive field (Carry forwards strong signals and ignores the weak ones). This layer is best used when we are trying to extract information related to distinct features.<br>
2. **Average Pooling Layer**: Takes the average of values present in it's receptive field (this carry forwards the general information of it's receptive field by averaging out all of the values). This layer is best used when we are trying to denoise and smoothen the signals present it's receptive field

## Step By Step implementation of Max Pooling Layer

In [1]:
import math
import torch

# Using the below input tensor and convolution filter for the entire step by step implementation of max pooling layer
input_tensor = torch.arange(0, 9, dtype=torch.float32).reshape(1, 1, 3, 3)
kernel_size = 2
stride = 1
padding = 0
dilation = 1

### 1. Unpacking the Input Dimensions
We extract the shape of the incoming feature map `input_tensor`. We need the batch size (batch_size), the number of channels (c_in), and the spatial dimensions (height h_in and width w_in) to correctly calculate our final output shape later.

In [2]:
batch_size, c_in, h_in, w_in = input_tensor.shape
print(f"BATCH SIZE: {batch_size} | INPUT CHANNELS: {c_in} | INPUT TENSOR HEIGHT: {h_in} | INPUT TENSOR WEIGHT: {w_in}")

BATCH SIZE: 1 | INPUT CHANNELS: 1 | INPUT TENSOR HEIGHT: 3 | INPUT TENSOR WEIGHT: 3


### 2. Unfolding the Tensor (Extracting Receptive Fields)
We use torch.nn.functional.unfold to extract all the sliding local blocks (receptive fields) from the input tensor. Instead of moving a window across the incoming input tensor using for loops unfold grabs every possible receptive field of specified kernel size, stride, padding, and dilation and flattens each receptive field into a single column vector. This transforms our spatial tensor into a 3D matrix of shape [batch_size, channels * kernel_size * kernel_size, total_number_of_patches].

In [3]:
input_tensor_unfolded = torch.nn.functional.unfold(input_tensor,
                                        kernel_size=kernel_size,
                                        padding=padding,
                                        stride=stride,
                                        dilation=dilation)
input_tensor_unfolded

tensor([[[0., 1., 3., 4.],
         [1., 2., 4., 5.],
         [3., 4., 6., 7.],
         [4., 5., 7., 8.]]])

### 3. Transposing the Patches for Aggregation
The output from unfold is slightly clunky for calculating the maximum or  patch. We transpose the last two dimensions (dim0=-2, dim1=-1) to swap the sequence. This rearranges our tensor so that each extracted patch (receptive field) is grouped together, making it straightforward to apply a reduction operation across the correct axis.

In [4]:
input_tensor_unfolded_transposed = input_tensor_unfolded.transpose(dim0=-2, dim1=-1)
input_tensor_unfolded_transposed

tensor([[[0., 1., 3., 4.],
         [1., 2., 4., 5.],
         [3., 4., 6., 7.],
         [4., 5., 7., 8.]]])

### 4. Applying the Pooling Operation
With the tensor correctly aligned we apply our pooling logic across the receptive field dimension. For Max Pooling: We use .max(dim=1).values to extract the single highest value from each local patch

In [5]:
max_values = input_tensor_unfolded_transposed.max(dim=1).values
max_values

tensor([[4., 5., 7., 8.]])

### 5. Calculating Output Spatial Dimensions
Before we can return the result we need to know the exact height (h_out) and width (w_out) of the new feature map. We calculate this using the standard spatial output formula for convolutions and pooling layers:$$H_{out} = \left\lfloor \frac{H_{in} + 2 \times \text{padding} - \text{dilation} \times (\text{kernel\_size} - 1) - 1}{\text{stride}} \right\rfloor + 1$$(The same formula applies for $W_{out}$ using $W_{in}$.)

In [6]:
h_out = math.floor((h_in + (2 * padding) - (dilation * (kernel_size - 1)) - 1) / stride) + 1
w_out = math.floor((w_in + (2 * padding) - (dilation * (kernel_size - 1)) - 1) / stride) + 1

h_out, w_out

(2, 2)

### 6. Reshaping to Final Spatial Format
Our pooled values are currently just a flat list of numbers. We use .reshape() to fold the tensor back into the standard PyTorch 4D format: [batch_size, channels, height, width], resulting in our final pooled feature map.

In [7]:
max_values.reshape(batch_size, c_in, h_out, w_out)

tensor([[[[4., 5.],
          [7., 8.]]]])

Similarly can be done for Avg Pool layer where instead of taking the max value we take average value.

To see the complete implementation you can have a look at <br>
`pytorch-fundamentals/src/pytorch_fundamentals/layers/pooling.py`

To see if the implementation is correct you can see the comparison between pytorch and the one we just saw at <br>
`pytorch-fundamentals/benchmarks/compare_pooling_functions.ipynb`